# PDE: change a prompt, watch a microwave rollout
Compare **“close the microwave”** with an instruction you write. The VLA weights,
initial simulator state, and inference seed stay fixed. Then ask a VLM to describe
the reference rollout and propose new instructions, and try one of them.

Run with JupyterLab on Linux and an NVIDIA GPU. This notebook targets **one A100, L4 or H100**
(native BF16), with at least **32 GB host RAM** recommended for model loading and
**40 GB free disk** for the environment and checkpoint. A free T4 runtime is not
supported by this BF16 path. Colab is also supported with a compatible GPU runtime.

The default is [RLinf’s public pi0.5 LIBERO SFT checkpoint](https://huggingface.co/RLinf/RLinf-Pi05-LIBERO-SFT),
not the paper's weak checkpoint. This is an interactive illustration; a reworded
prompt is not guaranteed to perform better. You can supply your own compatible
checkpoint. Rollouts use one environment and frozen-policy inference, without
starting a distributed RL job. The optional VLM uses OpenAI or a Qwen endpoint
hosted elsewhere, leaving the local GPU for the VLA.

**Validation:** the installer’s Python environment, real pi0.5 rollouts, Qwen
feedback, suggestion replay and video display code were tested on one H100 GPU.
Google-hosted Colab, its system-package installation and live OpenAI calls were
not exercised. See [recorded validation results](https://github.com/xinyunsunshine/PDE/blob/main/notebooks/microwave_validation.json).
Setup may take several minutes and downloads several GB.


## Saved results from the tested GPU run
These outputs replay artifacts from the real H100 run on October 8, 2026;
**they do not rerun inference or call an API**. They are retained so you can inspect
the videos and actual Qwen response without a GPU or API key. All three prompts
failed the simulator success condition. The VLM description is its interpretation.
The validation report below lists everything exercised and the remaining limits.

To regenerate these displays, launch Jupyter from this repository. Some GitHub
previews suppress embedded video; open the notebook locally to play it.

Download videos directly if your viewer hides them: [original](recorded/rollouts/reference.mp4), [manual rewrite](recorded/rollouts/custom.mp4), [VLM suggestion](recorded/vlm-rollouts/custom.mp4).


In [1]:
from pathlib import Path
import json
cwd = Path.cwd().resolve()
NOTEBOOKS = next(p / "notebooks" for p in (cwd, *cwd.parents)
                 if (p / "notebooks/microwave_validation.json").is_file())
print((NOTEBOOKS / "microwave_validation.json").read_text())


{
  "validation_date": "2026-10-08",
  "environment": {
    "platform": "Linux GPU cluster (not hosted Colab)",
    "gpu": "NVIDIA H100 80GB HBM3",
    "python": "3.11.14",
    "torch": "2.7.1",
    "transformers": "4.53.2",
    "system_packages": "Preinstalled host EGL/GL libraries; Colab apt-get step not exercised"
  },
  "checkpoint": {
    "repo_id": "RLinf/RLinf-Pi05-LIBERO-SFT",
    "revision": "45ccfcc4e28634f1576ebf78cab0fbe2fd82432d"
  },
  "rlinf_revision": "fce5435df9472e2c61957e4f849fb903fc70827c",
  "openpi_revision": "c5dc4b9296a1a4739bf52828f28a579f12dce763",
  "libero_revision": "0c5e40cc4ae63e09c14e7df6f74481e9ee8585f7",
  "task": "libero_90.close_the_microwave",
  "task_index": 33,
  "seed": 0,
  "trial": 0,
  "steps_per_rollout": 240,
  "comparisons": [
    {
      "prompt": "close the microwave",
      "simulator_success": false,
      "initial_image_sha256": "7d14ef85b57e9206c2db9d06181835a200353cfdf22cb8cb35b355589c5c70d2"
    },
    {
      "prompt": "push the mi

In [2]:
# 5. Watch both behaviors side by side
from IPython.display import HTML, display
import base64
import html

def show_comparison(directory):
    report = json.loads((directory / "comparison.json").read_text())
    panels = []
    for result in report["results"]:
        data = base64.b64encode((directory / result["video"]).read_bytes()).decode()
        panels.append(f"<div style='flex:1;min-width:240px'><h4>{html.escape(result['prompt'])}</h4>"
                      f"<p>Simulator success: {result['success']}</p>"
                      f"<video controls loop style='width:100%' src='data:video/mp4;base64,{data}'></video></div>")
    display(HTML("<div style='display:flex;gap:20px;flex-wrap:wrap'>" + "".join(panels) + "</div>"))
    print("Same initial image:", report["results"][0]["initial_image_sha256"] == report["results"][1]["initial_image_sha256"])

show_comparison(NOTEBOOKS / "recorded/rollouts")


Same initial image: True


In [3]:
recorded_feedback = json.loads((NOTEBOOKS / "recorded/rollouts/vlm_feedback.json").read_text())
print(json.dumps(recorded_feedback, indent=2))


{
  "provider": "local_qwen",
  "model": "Qwen/Qwen3-VL-2B-Instruct",
  "summary": "The robot attempted to close the microwave by placing a mug inside it, but failed to do so as it was not properly aligned and the microwave door did not close correctly.",
  "new_prompts": [
    "close the microwave door properly",
    "ensure the microwave door is fully closed",
    "close the microwave without a mug inside"
  ],
  "simulator_success": false
}


In [4]:
show_comparison(NOTEBOOKS / "recorded/vlm-rollouts")


Same initial image: True


## Run your own experiment
The following cells install dependencies and run new GPU inference. The optional
VLM cell makes live API calls. Their outputs are separate from the saved run above.

In [ ]:
# 1. Locate PDE and choose a workspace
import os
import sys
import json
import subprocess
from pathlib import Path

# Leave blank when launching Jupyter from PDE or its notebooks directory.
repo_directory = ""
workspace_directory = "~/pde-demo"
cwd = Path.cwd().resolve()
REPO = Path(repo_directory).expanduser().resolve() if repo_directory else next(
    (p for p in (cwd, *cwd.parents) if (p / "pde/demo.py").is_file()),
    Path.home() / "PDE",
)
WORKSPACE = Path(workspace_directory).expanduser().resolve()
WORKSPACE.mkdir(parents=True, exist_ok=True)
if not REPO.exists():
    subprocess.run(["git", "clone", "--recurse-submodules", "https://github.com/xinyunsunshine/PDE.git", str(REPO)], check=True)
if not (REPO / "pde/demo.py").is_file():
    raise ValueError("Set repo_directory to your PDE checkout")
subprocess.run(["git", "submodule", "update", "--init", "--recursive"], cwd=REPO, check=True)
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], check=True)
capabilities = subprocess.check_output(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], text=True)
if float(capabilities.strip().splitlines()[0]) < 8.0:
    raise RuntimeError("This pi0.5 path requires native BF16 (A100, L4 or H100)")
print("PDE revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())


In [ ]:
# 2. Install the isolated Python 3.11 inference environment
# Keep the notebook kernel unchanged. No restart is needed.
# Install Linux EGL/GL libraries and ffmpeg beforehand (see README).
install_env = dict(os.environ, PDE_DEMO_ENV=str(WORKSPACE / "env"),
                   PDE_DEMO_DEPS=str(WORKSPACE / "deps"))
subprocess.run(["bash", "scripts/install_demo.sh"], env=install_env, cwd=REPO, check=True)
PYTHON = str(WORKSPACE / "env/bin/python")
os.environ.update({"MUJOCO_GL": "egl", "PYOPENGL_PLATFORM": "egl",
                   "JAX_PLATFORMS": "cpu", "LIBERO_TYPE": "standard"})

def run_demo(*args):
    subprocess.run([PYTHON, "-m", "pde.demo", *map(str, args)], cwd=REPO, check=True)

subprocess.run([PYTHON, "-c", "import torch; print(torch.cuda.get_device_name()); assert torch.cuda.get_device_capability()[0] >= 8, 'Choose an A100 or L4 runtime'"], check=True)


In [ ]:
# 3. Choose and download the VLA checkpoint
checkpoint_id = "RLinf/RLinf-Pi05-LIBERO-SFT"
checkpoint_revision = "45ccfcc4e28634f1576ebf78cab0fbe2fd82432d"
# A compatible local checkpoint directory can be supplied instead of downloading.
local_checkpoint = ""
CHECKPOINT = str(Path(local_checkpoint).expanduser()) if local_checkpoint else str(WORKSPACE / "checkpoint")
if not local_checkpoint:
    script = """from huggingface_hub import HfApi, snapshot_download
from pathlib import Path
import json, sys
repo, revision, destination = sys.argv[1:]
sha = HfApi().model_info(repo, revision=revision, token=False).sha
snapshot_download(repo_id=repo, revision=sha, local_dir=destination, token=False)
Path(destination, 'demo_source.json').write_text(json.dumps({'repo_id': repo, 'revision': sha}))
print('Checkpoint revision:', sha)
"""
    subprocess.run([PYTHON, "-c", script, checkpoint_id, checkpoint_revision, CHECKPOINT], check=True)
print("Using checkpoint:", CHECKPOINT)


## Compare two prompts
Edit `custom_prompt` and run the next two cells again. Each comparison loads the
policy once, runs both prompts sequentially, and releases GPU memory when done.
`trial` selects the same benchmark initial state for both rollouts; `seed` fixes
the policy's random noise. Success is measured by the simulator's original
microwave task condition, regardless of the instruction supplied to the VLA.


In [ ]:
# 4. Write a prompt and run the comparison
custom_prompt = "push the microwave door until it shuts"
seed = 0
trial = 0
steps = 240
OUTPUT = WORKSPACE / "microwave-demo"
run_demo("compare", "--checkpoint", CHECKPOINT, "--prompt", custom_prompt,
         "--seed", seed, "--trial", trial, "--steps", steps, "--output", OUTPUT)


In [ ]:
# 5. Watch both behaviors side by side
from IPython.display import HTML, display
import base64
import html

def show_comparison(directory):
    report = json.loads((directory / "comparison.json").read_text())
    panels = []
    for result in report["results"]:
        data = base64.b64encode((directory / result["video"]).read_bytes()).decode()
        panels.append(f"<div style='flex:1;min-width:240px'><h4>{html.escape(result['prompt'])}</h4>"
                      f"<p>Simulator success: {result['success']}</p>"
                      f"<video controls loop style='width:100%' src='data:video/mp4;base64,{data}'></video></div>")
    display(HTML("<div style='display:flex;gap:20px;flex-wrap:wrap'>" + "".join(panels) + "</div>"))
    print("Same initial image:", report["results"][0]["initial_image_sha256"] == report["results"][1]["initial_image_sha256"])

show_comparison(OUTPUT)


## Ask a VLM what happened
This optional step makes **two API requests**: one with eight frames from the
reference rollout to summarize behavior, then one to propose three prompts from
that summary. It displays the **actual returned text**, not a saved or fabricated
example. An OpenAI call sends the sampled frames to OpenAI and incurs API usage.
Choose `local_qwen` for your own reachable OpenAI-compatible Qwen server; it should
run on another machine so the VLA GPU remains available for the VLA.

Set `OPENAI_API_KEY` or `QWEN_API_KEY` in the environment, or enter the key
through the hidden input. Keys are not written into the notebook or result files.


In [ ]:
# 6. Call the VLM and display its output
provider = "openai"
vlm_model = "gpt-4.1"
qwen_base_url = ""
# For local_qwen, set the server's model ID above (e.g. Qwen/Qwen3-VL-8B-Instruct).
from getpass import getpass
key_name = "OPENAI_API_KEY" if provider == "openai" else "QWEN_API_KEY"
api_key = os.environ.get(key_name) or getpass(
    f"{key_name} (local Qwen without authentication: leave blank): "
)
if provider == "openai" and not api_key:
    raise ValueError("An OpenAI API key is required for this optional step")
if provider == "local_qwen" and not qwen_base_url:
    raise ValueError("Enter the reachable Qwen server URL, including /v1")
args = ["feedback", "--output", OUTPUT, "--provider", provider, "--model", vlm_model]
if provider == "local_qwen":
    args += ["--base-url", qwen_base_url]
previous_key = os.environ.get(key_name)
try:
    if api_key:
        os.environ[key_name] = api_key
    else:
        os.environ.pop(key_name, None)
    run_demo(*args)
finally:
    if previous_key is None:
        os.environ.pop(key_name, None)
    else:
        os.environ[key_name] = previous_key
    del api_key
feedback = json.loads((OUTPUT / "vlm_feedback.json").read_text())
print("VLM summary:\n", feedback["summary"])
print("Suggested prompts:")
for i, prompt in enumerate(feedback["new_prompts"]):
    print(f"{i}: {prompt}")
print("Simulator success (ground truth):", feedback["simulator_success"])


In [ ]:
# 7. Try a VLM suggestion against the original prompt
suggestion_index = 0
if not 0 <= suggestion_index < len(feedback["new_prompts"]):
    raise ValueError("Choose an index printed in the previous cell")
suggested_prompt = feedback["new_prompts"][suggestion_index]
VLM_OUTPUT = WORKSPACE / "microwave-vlm-demo"
run_demo("compare", "--checkpoint", CHECKPOINT, "--prompt", suggested_prompt,
         "--seed", seed, "--trial", trial, "--steps", steps, "--output", VLM_OUTPUT)
show_comparison(VLM_OUTPUT)


## Keep your results
Videos, sampled frames, prompt text, simulator success and seed are saved under
`WORKSPACE / "microwave-demo"` and `WORKSPACE / "microwave-vlm-demo"`.
Save the notebook to keep displayed outputs. When using Colab, download your
results before disconnecting because its runtime storage is temporary.
A single matched pair illustrates behavior and is not a success-rate estimate.
If setup or inference fails, keep the full traceback and the package versions in
`WORKSPACE / "env/installed-requirements.txt"`. For CUDA memory errors, choose a
larger GPU; for host RAM exhaustion while loading, choose a high-memory runtime.

[Code](https://github.com/xinyunsunshine/PDE) ·
[RLinf](https://github.com/RLinf/RLinf) ·
[Colab runtime guidance](https://research.google.com/colaboratory/faq.html)
